# **Data continue**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
from google.colab import files

# Preprocesamiento y Balanceo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from imblearn.combine import SMOTETomek

# Modelos
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Métricas
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance

# --- CARGA DE DATOS ---
print("Por favor, sube el archivo 'telecom_x_limpio.csv' que descargaste en la Parte 1:")
uploaded = files.upload()

# Leemos el archivo subido usando io.BytesIO
df = pd.read_csv(io.BytesIO(uploaded['telecom_x_limpio.csv']))
print("\n¡Datos cargados correctamente! Dimensiones:", df.shape)

Por favor, sube el archivo 'telecom_x_limpio.csv' que descargaste en la Parte 1:


Limpieza y encoding

In [ ]:
# 1. Eliminar columnas irrelevantes
df = df.drop(columns=['ID_Cliente', 'Churn'], errors='ignore')

# 2. Separar variables explicativas (X) de la variable objetivo (y)
X = df.drop('Churn_Binario', axis=1)
y = df['Churn_Binario']

# 3. Identificar columnas categóricas y numéricas
cols_cat = X.select_dtypes(include='object').columns.tolist()
cols_num = X.select_dtypes(exclude='object').columns.tolist()

# 4. ColumnTransformer: OrdinalEncoder para categóricas, passthrough para numéricas
#    OrdinalEncoder asigna un entero por categoría sin expandir columnas (alternativa a OHE)
preprocessor = ColumnTransformer(transformers=[
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cols_cat)
], remainder='passthrough')

X_encoded_arr = preprocessor.fit_transform(X)

# Reconstruir nombres de columnas
all_cols = cols_cat + cols_num
X_encoded = pd.DataFrame(X_encoded_arr, columns=all_cols, index=X.index)

print(f"Dimensiones de X procesado (OrdinalEncoder): {X_encoded.shape}")
print(f"Proporción de Churn (Cancelación) actual:\n{y.value_counts(normalize=True)*100}")

Análisis de Correlación y Gráficos Dirigidos

In [ ]:
from sklearn.feature_selection import chi2

# Chi-cuadrado entre cada variable discreta y el objetivo (requiere valores >= 0)
mask = y.notna()
X_chi = X_encoded[mask].fillna(0).clip(lower=0)   # chi2 necesita valores no negativos
chi2_scores, p_values = chi2(X_chi, y[mask])
chi2_series = pd.Series(chi2_scores, index=X_encoded.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
top_chi2 = chi2_series.head(11)
sns.barplot(x=top_chi2.values, y=top_chi2.index, palette='magma')
plt.title('Relevancia de Variables con la Cancelación (Churn) — Chi-Cuadrado')
plt.xlabel('Estadístico Chi²')
plt.tight_layout()
plt.show()

# Gráficos dirigidos (Box plots con swarm)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='Churn_Binario', y='Meses_Contrato', data=df, ax=axes[0], palette='Set2')
sns.stripplot(x='Churn_Binario', y='Meses_Contrato', data=df, ax=axes[0],
              color='black', alpha=0.15, size=2, jitter=True)
axes[0].set_title('Meses de Contrato vs Cancelación')

sns.boxplot(x='Churn_Binario', y='Cobro_Total', data=df, ax=axes[1], palette='Set2')
sns.stripplot(x='Churn_Binario', y='Cobro_Total', data=df, ax=axes[1],
              color='black', alpha=0.15, size=2, jitter=True)
axes[1].set_title('Cobro Total vs Cancelación')
plt.show()

Separación, Balanceo (SMOTETomek) y Normalización

In [ ]:
# --------------------------------------------------------
# PARCHE DE SEGURIDAD: Eliminar valores nulos (NaN)
# --------------------------------------------------------
# 1. Conservamos solo las filas donde 'y' NO es nulo
mascara_no_nulos = y.notna()
X_encoded = X_encoded[mascara_no_nulos]
y = y[mascara_no_nulos]

# 2. Por seguridad, rellenamos cualquier nulo accidental en X con 0
X_encoded = X_encoded.fillna(0)

print(f"Borrados los valores nulos. Nuevas dimensiones de X: {X_encoded.shape}")
print(f"Nuevas dimensiones de y: {y.shape}")
print("-" * 50)

# --------------------------------------------------------
# AHORA SÍ: Separación, Balanceo (SMOTETomek) y Normalización
# --------------------------------------------------------
# 1. División Train/Test (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.30, random_state=42, stratify=y
)

# 2. Balanceo con SMOTETomek (sobre-muestrea la clase minoritaria Y elimina ruido frontera)
smt = SMOTETomek(random_state=42)
X_train_bal, y_train_bal = smt.fit_resample(X_train, y_train)
X_train_bal = pd.DataFrame(X_train_bal, columns=X_encoded.columns)
y_train_bal = pd.Series(y_train_bal)

# 3. Normalización / Escalado RobustScaler (resistente a outliers)
scaler = RobustScaler()

# Lista de tus columnas estrictamente numéricas (las que no son 0 y 1)
cols_numericas = ['Meses_Contrato', 'Cobro_Mensual', 'Cobro_Total', 'Cuentas_Diarias', 'Cant_Servicios']

# Ajustamos el escalador con el entrenamiento y transformamos ambos
X_train_bal[cols_numericas] = scaler.fit_transform(X_train_bal[cols_numericas])
X_test[cols_numericas] = scaler.transform(X_test[cols_numericas])

print(f"Distribución de Churn en Entrenamiento después de SMOTETomek:\n{y_train_bal.value_counts()}")

Entrenamiento y Evaluación de Modelos

In [ ]:
# --- ENTRENAMIENTO ---
# Modelo 1: Support Vector Machine (SVM)
modelo_svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
modelo_svm.fit(X_train_bal, y_train_bal)
y_pred_svm = modelo_svm.predict(X_test)

# Modelo 2: K-Nearest Neighbors
modelo_knn = KNeighborsClassifier(n_neighbors=7, metric='manhattan')
modelo_knn.fit(X_train_bal, y_train_bal)
y_pred_knn = modelo_knn.predict(X_test)

# --- EVALUACIÓN (Función auxiliar) ---
def reporte_modelo(nombre, y_verdadero, y_predicho, modelo=None, X_test_df=None):
    print(f"\n{'='*50}")
    print(f"REPORTE DE RENDIMIENTO: {nombre}")
    print(f"{'='*50}")
    print(classification_report(y_verdadero, y_predicho))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))

    # Matriz de confusión
    sns.heatmap(confusion_matrix(y_verdadero, y_predicho), annot=True, fmt='d',
                cmap='Blues', ax=axes[0])
    axes[0].set_title(f'Matriz de Confusión - {nombre}')
    axes[0].set_ylabel('Real')
    axes[0].set_xlabel('Predicho')

    # Curva ROC
    if modelo is not None and X_test_df is not None:
        y_proba = modelo.predict_proba(X_test_df)[:, 1]
        fpr, tpr, _ = roc_curve(y_verdadero, y_proba)
        auc = roc_auc_score(y_verdadero, y_proba)
        axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc:.3f}')
        axes[1].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
        axes[1].set_title(f'Curva ROC - {nombre}')
        axes[1].set_xlabel('Tasa de Falsos Positivos')
        axes[1].set_ylabel('Tasa de Verdaderos Positivos')
        axes[1].legend(loc='lower right')

    plt.tight_layout()
    plt.show()

reporte_modelo("SVM (KERNEL RBF)", y_test, y_pred_svm, modelo_svm, X_test)
reporte_modelo("K-NEAREST NEIGHBORS", y_test, y_pred_knn, modelo_knn, X_test)

Interpretación Estratégica

In [ ]:
# Importancia de Variables en KNN via Permutation Importance
result = permutation_importance(modelo_knn, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importancias = pd.DataFrame({
    'Variable': X_train_bal.columns,
    'Importancia': result.importances_mean
}).sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=importancias.head(10), palette='crest')
plt.title('Top 10 Variables más Importantes (KNN — Permutation Importance)')
plt.show()

# Coeficientes del SVM via Permutation Importance (SVM con kernel RBF no tiene coef_ directos)
result_svm = permutation_importance(modelo_svm, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
coef_svm = pd.DataFrame({
    'Variable': X_train_bal.columns,
    'Importancia': result_svm.importances_mean
}).sort_values(by='Importancia', ascending=False)

print("\n--- INSIGHTS ESTRATÉGICOS (SVM) ---")
print("\nFactores que MÁS AUMENTAN la probabilidad de cancelar:")
print(coef_svm.head(5))

print("\nFactores que MÁS AYUDAN A RETENER al cliente:")
print(coef_svm.tail(5))